In [23]:
import pandas as pd
import numpy as np

In [3]:
nav = pd.read_csv('data/processed/nav_history_clean.csv', parse_dates=['date'])
nav.head()

,amfi_code,date,nav
0,100016,2022-01-03,520.4608
1,100016,2022-01-04,515.0971
2,100016,2022-01-05,521.7239
3,100016,2022-01-06,515.7880
4,100016,2022-01-07,515.1639


In [4]:
nav = nav.sort_values(['amfi_code', 'date']).reset_index(drop=True)

In [6]:
nav

,amfi_code,date,nav
0,100016,2022-01-03,520.4608
1,100016,2022-01-04,515.0971
2,100016,2022-01-05,521.7239
3,100016,2022-01-06,515.7880
4,100016,2022-01-07,515.1639
...,...,...,...
45995,149324,2026-05-25,292.4810
45996,149324,2026-05-26,291.2707
45997,149324,2026-05-27,288.8007
45998,149324,2026-05-28,280.6873


In [7]:
nav['daily_return'] = nav.groupby('amfi_code')['nav'].pct_change()

In [8]:
nav['daily_return'].describe()

count    45960.000000
mean         0.000631
std          0.010290
min         -0.058102
25%         -0.005042
50%          0.000340
75%          0.006324
max          0.064713
Name: daily_return, dtype: float64

In [10]:
fund_master = pd.read_csv('data/raw/01_fund_master.csv')
fund_master.head()

,amfi_code,fund_house,scheme_name,category,sub_category,plan,launch_date,benchmark,expense_ratio_pct,exit_load_pct,min_sip_amount,min_lumpsum_amount,fund_manager,risk_category,sebi_category_code
0,119551,SBI Mutual Fund,SBI Bluechip Fund - Regular Plan - Growth,Equity,Large Cap,Regular,2006-02-14,NIFTY 100 TRI,1.54,1.0,500,1000,Sohini Andani,Moderate,EC01
1,119552,SBI Mutual Fund,SBI Bluechip Fund - Direct Plan - Growth,Equity,Large Cap,Direct,2013-01-01,NIFTY 100 TRI,0.66,1.0,500,1000,Sohini Andani,Moderate,EC01
2,119598,SBI Mutual Fund,SBI Small Cap Fund - Regular Plan - Growth,Equity,Small Cap,Regular,2009-09-09,BSE 250 SmallCap TRI,1.43,1.0,500,1000,R. Srinivasan,Very High,EC03
3,119599,SBI Mutual Fund,SBI Small Cap Fund - Direct Plan - Growth,Equity,Small Cap,Direct,2013-01-01,BSE 250 SmallCap TRI,0.72,1.0,500,1000,R. Srinivasan,Very High,EC03
4,119120,SBI Mutual Fund,SBI Magnum Gilt Fund - Regular Plan - Growth,Debt,Gilt,Regular,2000-12-30,CRISIL Dynamic Gilt Index,0.77,0.0,500,1000,Dinesh Ahuja,Low,DC02


In [11]:
nav_with_cat = nav.merge(fund_master[['amfi_code', 'category']], on='amfi_code')

In [12]:
nav_with_cat.groupby('category')['daily_return'].std().sort_values(ascending=False)

category
Equity    0.011135
Debt      0.001775
Name: daily_return, dtype: float64

In [14]:
latest_date = nav['date'].max()
latest_date

Timestamp('2026-05-29 00:00:00')

In [19]:
def cagr_for_period(group, years_back):
    end_date = group['date'].max()
    start_date = end_date - pd.DateOffset(years=years_back)
    if group['date'].min() > start_date:
        return None
    start_row = group[group['date'] >= start_date].sort_values('date').iloc[0]
    end_row = group[group['date'] == end_date].iloc[0]
    n_years = (end_row['date'] - start_row['date']).days / 365.25
    return (end_row['nav'] / start_row['nav']) ** (1 / n_years) - 1

In [20]:
def cagr_since_inception(group):
    start_row = group.sort_values('date').iloc[0]
    end_row = group.sort_values('date').iloc[-1]
    n_years = (end_row['date'] - start_row['date']).days / 365.25
    cagr = (end_row['nav'] / start_row['nav']) ** (1 / n_years) - 1
    return cagr, n_years

In [21]:
results = []
for code, group in nav.groupby('amfi_code'):
    since_inception_cagr, n_years = cagr_since_inception(group)
    results.append({
        'amfi_code': code,
        'return_1yr_pct': cagr_for_period(group, 1),
        'return_3yr_pct': cagr_for_period(group, 3),
        'return_since_inception_pct': since_inception_cagr,
        'since_inception_n_years': round(n_years, 2),   # e.g. 4.41 -- proves it's honest
    })

In [22]:
cagr_df = pd.DataFrame(results)
cagr_df[['return_1yr_pct', 'return_3yr_pct', 'return_since_inception_pct']] *= 100
cagr_df

,amfi_code,return_1yr_pct,return_3yr_pct,return_since_inception_pct,since_inception_n_years
0,100016,-2.225777,1.292353,2.637074,4.4
1,100025,3.707553,3.915479,4.458210,4.4
2,100033,53.277195,32.433971,30.123153,4.4
3,101206,47.963794,28.960211,23.538361,4.4
4,101207,-24.000309,-4.151454,7.938765,4.4
5,101208,7.241777,6.314299,6.509027,4.4
6,102885,20.222859,19.662361,18.232379,4.4
7,102886,-16.807960,-0.767232,1.171745,4.4
8,102887,13.593044,25.549670,16.841120,4.4
9,118632,34.007896,22.646647,24.049493,4.4


In [25]:
RISK_FREE_RATE = 0.065
TRADING_DAYS = 252
daily_rf = RISK_FREE_RATE / TRADING_DAYS

In [26]:
sharpe_results = []
for code, group in nav.groupby('amfi_code'):
    returns = group['daily_return'].dropna()
    mean_daily_return = returns.mean()
    std_daily_return = returns.std()
    sharpe = (mean_daily_return - daily_rf) / std_daily_return * np.sqrt(TRADING_DAYS)
    sharpe_results.append({'amfi_code': code, 'sharpe_ratio': sharpe})

In [27]:
sharpe_df = pd.DataFrame(sharpe_results)
sharpe_df['rank'] = sharpe_df['sharpe_ratio'].rank(ascending=False).astype(int)
sharpe_df = sharpe_df.sort_values('sharpe_ratio', ascending=False).reset_index(drop=True)
sharpe_df

,amfi_code,sharpe_ratio,rank
0,148567,1.448291,1
1,120843,1.306744,2
2,148569,1.234930,3
3,119551,1.208267,4
4,120505,1.180101,5
5,149323,1.132122,6
6,100033,1.093699,7
7,118632,1.081659,8
8,101206,1.027213,9
9,120504,1.026524,10


In [28]:
sortino_results = []
for code, group in nav.groupby('amfi_code'):
    returns = group['daily_return'].dropna()
    mean_daily_return = returns.mean()

    downside_returns = returns[returns < 0]
    downside_std = downside_returns.std()

    sortino = (mean_daily_return - daily_rf) / downside_std * np.sqrt(TRADING_DAYS)
    sortino_results.append({'amfi_code': code, 'sortino_ratio': sortino, 'n_negative_days': len(downside_returns)})

In [29]:
sortino_df = pd.DataFrame(sortino_results)
sortino_df['rank'] = sortino_df['sortino_ratio'].rank(ascending=False).astype(int)
sortino_df = sortino_df.sort_values('sortino_ratio', ascending=False).reset_index(drop=True)
sortino_df

,amfi_code,sortino_ratio,n_negative_days,rank
0,148567,2.385644,506,1
1,120843,2.364320,534,2
2,148569,2.146914,545,3
3,119551,2.140267,518,4
4,120505,2.029353,539,5
5,149323,1.875101,526,6
6,118632,1.850133,532,7
7,100033,1.829134,538,8
8,120504,1.805294,517,9
9,101206,1.799563,539,10
